In [ ]:
from pathlib import Path
import os
import sys

# Find the repository root when launched from either the notebook folder or its parent.
repo_path = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "code/vae_convlstm/paths.py").is_file()),
    None,
)
if repo_path is None:
    raise FileNotFoundError("Open this notebook from the repository or a subfolder of it")
sys.path.insert(0, str(repo_path / "code/vae_convlstm"))
from paths import TEXAS_DATA_ROOT, TEXAS_TRAIN_DIR, TEXAS_TEST_DIR, VAE_CHECKPOINT_DIR, VAE_FIGURE_DIR


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import einops
from utils import set_seed
import matplotlib.pyplot as plt
from train_utils import BatchIndicesIterator, plot_reconstruction_reference_error, load_latest_checkpoint, detect_outliers, limited_gradient
from utils import set_seed, RelativeL2Loss, format_elapsed_time, print_model_size

In [ ]:
set_seed(42)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
maeloss = torch.nn.L1Loss(reduction='sum')
mseloss = torch.nn.MSELoss(reduction='sum')


class CustomDataset(Dataset):
    def __init__(self, x_data, device):
        self.x_data = x_data
        self.x_data["y"] = torch.Tensor(self.x_data["y"])
        self.x_data["u"] = torch.Tensor(self.x_data["u"])
        self.device = device
    def __len__(self):
        return len(self.x_data["y"])

    def __getitem__(self, idx):
        return self.x_data["y"][idx].to(self.device), self.x_data["u"][idx].to(self.device)

def loss_function(x, x_hat, mean, log_var, mean_dynamics, beta=0.0, lambda_=1.0):
    reconstruction_loss = mseloss(x_hat[:,:,0], x[:,:, 0]) + lambda_ * (mseloss(x_hat[:,:, 1], x[:,:, 1]) + mseloss(x_hat[:,:, 2], x[:,:,2])) + mseloss(x_hat[:,:, 3], x[:,:,3]) + mseloss(mean[:, 1:], mean_dynamics[:, :-1])
    # KLD = - 0.5 * torch.sum(1+ log_var - mean.pow(2) - log_var.exp())

    return reconstruction_loss# + beta * KLD

def kld(mean1, log_var1, mean2, log_var2):
    return - 0.5 * torch.sum(1+ log_var1 - log_var2 - (mean1 - mean2).pow(2)/log_var2.exp() - log_var1.exp()/log_var2.exp())

In [ ]:
from normalization import Normalize_gaussian
required_stats = (VAE_CHECKPOINT_DIR / "mean_std.npz", VAE_CHECKPOINT_DIR / "mean_std_rain_source.npz")
missing_stats = [str(path) for path in required_stats if not path.is_file()]
if missing_stats:
    raise FileNotFoundError("Missing VAE normalization files: " + ", ".join(missing_stats))
dataset_directory = TEXAS_TRAIN_DIR
sample_start_index = 10
sample_end_index   = 11
burn_in_length = 1

rain_source = [None] * len(range(sample_start_index, sample_end_index + 1))
flow_variables = [None] * len(range(sample_start_index, sample_end_index + 1))
for i in tqdm(range(sample_start_index, sample_end_index + 1)):
    sample_directory = os.path.join(dataset_directory, f"sample_{i:05d}")
    rain_source[i - sample_start_index] = (np.load(os.path.join(sample_directory, "rain_source.npy")))
    flow_variables[i - sample_start_index] = np.load(os.path.join(sample_directory, "flow_variables.npy"))
    rain_source[i - sample_start_index] = np.repeat(rain_source[i - sample_start_index][:-1, 1], 4)
flow_variables = np.array(flow_variables).astype(np.float32)

burn_in_length = 1
flow_variables = flow_variables[:, burn_in_length:, :, :]
rain_source = np.array(rain_source).astype(np.float32)[:, :, np.newaxis]
num_time_instants = flow_variables.shape[1]
stat = np.load(VAE_CHECKPOINT_DIR / "mean_std.npz")
flow_mean = stat["mean"]
flow_std = stat["std"]
normalize_y = Normalize_gaussian(flow_mean, flow_std)
flow_variables = normalize_y.normalize_forw(flow_variables)
stat = np.load(VAE_CHECKPOINT_DIR / "mean_std_rain_source.npz")
rain_mean = stat["mean"]
rain_std = stat["std"]
normalize_u = Normalize_gaussian(rain_mean, rain_std)
rain_source = normalize_u.normalize_forw(rain_source)
coords = np.repeat(np.stack([np.repeat(2*(np.arange(0, flow_variables.shape[3], 1)/flow_variables.shape[3] - 0.5), flow_variables.shape[4]), np.tile(2*(np.arange(0, flow_variables.shape[4], 1)/flow_variables.shape[4] - 0.5), flow_variables.shape[3])], axis = 1)[None, :, :], flow_variables.shape[0], axis = 0).astype(np.float32)
data_train = {"u": rain_source, "y":flow_variables}


dataset_directory = TEXAS_TEST_DIR
sample_start_index = 101
sample_end_index   = 120
burn_in_length = 1

rain_source = [None] * len(range(sample_start_index, sample_end_index + 1))
flow_variables = [None] * len(range(sample_start_index, sample_end_index + 1))
for i in tqdm(range(sample_start_index, sample_end_index + 1)):
    sample_directory = os.path.join(dataset_directory, f"sample_{i:05d}")
    rain_source[i - sample_start_index] = (np.load(os.path.join(sample_directory, "rain_source.npy")))
    flow_variables[i - sample_start_index] = np.load(os.path.join(sample_directory, "flow_variables.npy"))
    rain_source[i - sample_start_index] = np.repeat(rain_source[i - sample_start_index][:-1, 1], 4)
flow_variables = np.array(flow_variables).astype(np.float32)
burn_in_length = 1
flow_variables = flow_variables[:, burn_in_length:, :, :]
rain_source = np.array(rain_source).astype(np.float32)[:, :, np.newaxis]
flow_variables = normalize_y.normalize_forw(flow_variables)
rain_source = normalize_u.normalize_forw(rain_source)
data_valid = {"u": rain_source, "y":flow_variables}

data_train["u"] = np.zeros((data_train["u"].shape[0], 192, 1, 25, 25)) + data_train["u"][:, :, :, np.newaxis, np.newaxis]
data_valid["u"] = np.zeros((data_valid["u"].shape[0], 192, 1, 25, 25)) + data_valid["u"][:, :, :, np.newaxis, np.newaxis]

In [ ]:
train_dataset = CustomDataset(data_train, device)
valid_dataset = CustomDataset(data_valid, device)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=1, shuffle=False)


In [ ]:
from ldm_ae.convlstm import *
vae_model_file = repo_path / "code/vae_convlstm/vae_texas.py"
if not vae_model_file.is_file():
    raise FileNotFoundError(f"Missing VAE_LSTM model definition: {vae_model_file}")
from vae_texas import VAE_LSTM
model = VAE_LSTM(
    resolution=256, 
    in_channel=3, 
    ch_mult=(1,4,16),
    num_res_blocks=3, 
    hidden_dim=16, 
    embed_dim=8,
    latent_dim = 256
)

model = model.to(device)

_, _ = print_model_size(model)

In [ ]:
checkpoint_dir = VAE_CHECKPOINT_DIR
checkpoint = torch.load(checkpoint_dir / "checkpoint_vae.pth", map_location=device, weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
start_epoch = checkpoint['epoch'] + 1
epoch_loss_list = checkpoint['epoch_loss_list']
frames = checkpoint.get('frames', [])
checkpoint_epoch = checkpoint['epoch']

# The VAE checkpoint includes its own dynamics weights.
model.eval()

In [ ]:
# def loss_function(x, x_hat, mean, log_var, mean_dynamics, beta=0.0, lambda_=1.0):
#     reconstruction_loss = mseloss(x_hat[:,:,0], x[:,:, 0]) + lambda_ * (mseloss(x_hat[:,:, 1], x[:,:, 1]) + mseloss(x_hat[:,:, 2], x[:,:,2])) + mseloss(x_hat[:,:, 3], x[:,:,3]) + mseloss(mean[:, 1:], mean_dynamics[:, :-1])
#     # KLD = - 0.5 * torch.sum(1+ log_var - mean.pow(2) - log_var.exp())

#     return reconstruction_loss# + beta * KLD
# num_states = 20
# predicted_states = np.zeros(shape = (1, 96, 4, 256, 256))
# target_states = np.zeros(shape = (1, 96, 4, 256, 256))
# latent_states = torch.zeros(1, 192, 8, 25, 25).to(device)
# given = 30
# x_hat = 0
# for i, value in enumerate(valid_loader):
#     if i == 0:
#         x = value
#         target, rain = x
#         with torch.no_grad():
#             x_hat, mean, log_var, mean_dynamics = model((target[:, 0:given, :, :, :], rain[:, 0:given, :, 0, 0]))
            
#             latent_states[:, 0:given, :] = mean
#             for j in range(given, 96):
#                 #residual version or not
#                 # x_hat, mean, log_var, mean_dynamics = model((target[:, 0:given, :, :, :], rain[:, 0:given, :, 0, 0]))
#                 latent_states[:, j] = model.dynamics(torch.cat([latent_states[:, :j], rain[:, :j]], dim = 2))[0][2][:, j-1] + latent_states[:, j-1]
#                 # latent_states[:, j] = model.propagate_dynamics(latent_states[:, :j])[0][0][:, j-1]
# with torch.no_grad():
#     latent_states = latent_states.reshape([latent_states.shape[0]*latent_states.shape[1], -1, 25, 25])
#     dec_array = []
#     for i in range(latent_states.shape[0]):
#         dec_array.append(model.decode(latent_states[i:i+1]))
#     dec = torch.stack(dec_array, dim = 1)
#     predicted_states[i] = dec.detach().cpu().numpy()
#     target_states[i] = target.detach().cpu().numpy()

In [ ]:
def loss_function(x, x_hat, mean, log_var, mean_dynamics, beta=0.0, lambda_=1.0):
    reconstruction_loss = mseloss(x_hat[:,:,0], x[:,:, 0]) + lambda_ * (mseloss(x_hat[:,:, 1], x[:,:, 1]) + mseloss(x_hat[:,:, 2], x[:,:,2])) + mseloss(x_hat[:,:, 3], x[:,:,3]) + mseloss(mean[:, 1:], mean_dynamics[:, :-1])
    # KLD = - 0.5 * torch.sum(1+ log_var - mean.pow(2) - log_var.exp())

    return reconstruction_loss# + beta * KLD
num_states = 20
latent_states = torch.zeros(1, 192, 8, 25, 25).to(device)
predicted_states = np.zeros(shape = (num_states, 192, 3, 500, 500))
target_states = np.zeros(shape = (num_states, 192, 3, 500, 500))
given = 1
x_hat = 0
for i, value in enumerate(valid_loader):
    if i < num_states:
        x = value
        target, rain = x
        print(target.shape)
        with torch.no_grad():
            x_hat, mean, log_var, mean_dynamics = model((target[:, 0:given, :, :, :], rain[:, 0:given, :, 0, 0]))
            
            latent_states[:, 0:given, :] = mean
            for j in range(given, 192):
                #residual version or not
                latent_states[:, j] = model.dynamics(torch.cat([latent_states[:, :j], rain[:, :j]], dim = 2))[0][2][:, j-1] + latent_states[:, j-1]
                # latent_states[:, j] = model.propagate_dynamics(latent_states[:, :j])[0][0][:, j-1]
        with torch.no_grad():
            decoded_steps = []
            for t in range(latent_states.shape[1]):
                dec_t = model.decode(latent_states[:, t])   # shape: [B, 4, 500, 500]
                decoded_steps.append(dec_t)
            dec = torch.stack(decoded_steps, dim=1)         # shape: [B, T, 4, 500, 500]

        predicted_states[i] = dec.detach().cpu().numpy()
        target_states[i] = target.detach().cpu().numpy()

In [ ]:
def summarize_reconstruction_metrics(name, prediction, ground_truth, channel_names):
    prediction = normalize_y.normalize_backward(np.asarray(prediction, dtype=np.float64))[:, :, :len(channel_names)]
    ground_truth = normalize_y.normalize_backward(np.asarray(ground_truth, dtype=np.float64))[:, :, :len(channel_names)]

    rmse_by_channel = np.sqrt(np.mean((prediction - ground_truth) ** 2, axis=(0, 1, 3, 4)))
    normalization = np.sqrt(np.mean((ground_truth) ** 2, axis=(0, 1, 3, 4)))
    r2_by_channel = np.array([
        r2_score(ground_truth[:, :, channel].reshape(-1), prediction[:, :, channel].reshape(-1))
        for channel in range(len(channel_names))
    ])
    rel_rmse_by_channel = rmse_by_channel / normalization

    print(f"{name} average rel RMSE: {rel_rmse_by_channel.mean():.6f}")
    print(f"{name} average RMSE: {rmse_by_channel.mean():.6f}")
    print(f"{name} average R^2: {r2_by_channel.mean():.6f}")
    print(f"{name} rel RMSE by channel: {dict(zip(channel_names, rel_rmse_by_channel))}")
    print(f"{name} RMSE by channel: {dict(zip(channel_names, rmse_by_channel))}")
    print(f"{name} R^2 by channel: {dict(zip(channel_names, r2_by_channel))}")

    return rmse_by_channel, r2_by_channel

channel_names = ("h", "hu", "hv")
summarize_reconstruction_metrics("Train", predicted_states, target_states, channel_names)


In [ ]:
print("Train RMSE by channel:", dict(zip(channel_names, train_rmse_by_channel)))
print("Train R^2 by channel:", dict(zip(channel_names, train_r2_by_channel)))

In [ ]:
# Train metrics are printed in the previous cell.

In [ ]:
def loss_function(x, x_hat, mean, log_var, mean_dynamics, beta=0.0, lambda_=1.0):
    reconstruction_loss = mseloss(x_hat[:,:,0], x[:,:, 0]) + lambda_ * (mseloss(x_hat[:,:, 1], x[:,:, 1]) + mseloss(x_hat[:,:, 2], x[:,:,2])) + mseloss(x_hat[:,:, 3], x[:,:,3]) + mseloss(mean[:, 1:], mean_dynamics[:, :-1])
    # KLD = - 0.5 * torch.sum(1+ log_var - mean.pow(2) - log_var.exp())

    return reconstruction_loss# + beta * KLD
num_states = len(valid_loader)
latent_dim = 8
state_dim = 4
latent_states = torch.zeros(1, 96, latent_dim, 16, 16).to(device)
predicted_states = np.zeros(shape = (num_states, 96, state_dim, 256, 256))
target_states = np.zeros(shape = (num_states, 96, state_dim, 256, 256))
given = 1
x_hat = 0
for i, value in enumerate(valid_loader):
    if i < num_states:
        x = value
        target = x
        with torch.no_grad():
            x_hat, mean, log_var, mean_dynamics = model(target[:, 0:given, :, :, :])
            
            latent_states[:, 0:given, :] = mean
            for j in range(given, 96):
                #residual version or not
                latent_states[:, j] = latent_states[:, j-1] + model.propagate_dynamics(latent_states[:, :j])[:, j-1]
                # latent_states[:, j] = model.propagate_dynamics(latent_states[:, :j])[0][0][:, j-1]
        with torch.no_grad():
            dec = model.decode(latent_states.reshape([latent_states.shape[0]*latent_states.shape[1], -1, 16, 16])).reshape(1, 96, state_dim, 256, 256)
        predicted_states[i] = dec.detach().cpu().numpy()
        target_states[i] = target.detach().cpu().numpy()

In [ ]:
channel_names = ("h", "hu", "hv")
test_rmse_by_channel, test_r2_by_channel = summarize_reconstruction_metrics("Test", predicted_states, target_states, channel_names)

In [ ]:
print("Test RMSE by channel:", dict(zip(channel_names, test_rmse_by_channel)))
print("Test R^2 by channel:", dict(zip(channel_names, test_r2_by_channel)))

In [ ]:
# results = []
# orig_data = []
# for i, value in enumerate(train_loader):
#     x = value
#     target = x
#     with torch.no_grad():
#         x_hat, mean, log_var, mean_dynamics = model(target[:, :, :, :, :])
#         results.append(mean.detach().cpu().numpy())
#         orig_data.append(target.detach().cpu().numpy())
# data = {}
# data["latent_train"] = np.concatenate(results, axis = 0)
# data["state_train"] = np.concatenate(orig_data, axis = 0)
# results = []
# orig_data = []
# for i, value in enumerate(valid_loader):
#     x = value
#     target = x
#     with torch.no_grad():
#         x_hat, mean, log_var, mean_dynamics = model(target[:, :, :, :, :])
#         results.append(mean.detach().cpu().numpy())
#         orig_data.append(target.detach().cpu().numpy())
# data["latent_test"] = np.concatenate(results, axis = 0)
# data["state_test"] = np.concatenate(orig_data, axis = 0)


In [ ]:
# print(mean[0, 1:] - mean_dynamics[0, :-1])
# print(mean_dynamics[0, :-1])
# print(mean[0, 1:])
# print(mseloss(mean[:, 1:], mean_dynamics[:, :-1]))
# print(mseloss(x_hat[:,:,0], x[:,:, 0]))
# print(mseloss(x_hat[:,:, 1], x[:,:, 1]))
# print(mseloss(x_hat[:,:, 2], x[:,:,2]))
# print(mseloss(x_hat[:,:, 3], x[:,:,3]))

In [ ]:
print(np.mean((dec.detach().cpu().numpy() - target.detach().cpu().numpy())**2, axis = (0, 2, 3, 4)))
#this is for original

In [ ]:
print(np.mean((dec.detach().cpu().numpy() - target.detach().cpu().numpy())**2, axis = (0, 2, 3, 4)))
#this is for convlstm

In [ ]:
print(np.mean((dec.detach().cpu().numpy() - target.detach().cpu().numpy())**2, axis = (0, 2, 3, 4)))
#this is for convlstm finetune with residual

In [ ]:
def compute_relative_l2_error(prediction: np.ndarray, ground_truth: np.ndarray):
    """
    Compute the relative L2 error between predictions and labels using NumPy.

    Inputs:
    predictions: (batch_size, num_timestamps, num_features, height, width)
    labels: (batch_size, num_timestamps, num_features, height, width)

    Outputs:
    relative_error: (batch_size, num_timestamps, num_features)
    """
    prediction = np.asarray(prediction, dtype=np.float64)
    ground_truth = np.asarray(ground_truth, dtype=np.float64)

    diff_norm = np.sqrt(np.sum((prediction - ground_truth) ** 2, axis=(3, 4)))
    gt_norm = np.sqrt(np.sum(ground_truth ** 2, axis=(3, 4)))

    # gt_norm = np.where(gt_norm == 0, 1e-12, gt_norm)

    return diff_norm / gt_norm


import io
import imageio

colorbar_vmin = {
    'water_depth': 0.0,
    'discharge_x': -15.0,
    'discharge_y': -15.0
}
colorbar_vmax = {
    'water_depth': 40.0,
    'discharge_x': 15.0,
    'discharge_y': 15.0
}


In [ ]:
figure_dir = VAE_FIGURE_DIR
figure_dir.mkdir(parents=True, exist_ok=True)

from train_utils import plot_prediction_truth_error
relative_l2_error = compute_relative_l2_error(dec.detach().cpu().numpy(), target.detach().cpu().numpy())
avg_relative_l2_error = np.mean(relative_l2_error, axis=0)

frames = []
for time_instant in range(1, num_time_instants + 1):
    time_index = time_instant - 1
    fig = plot_prediction_truth_error(dec[0, time_index, 0].detach().cpu().numpy(),
                                    target[0, time_index, 0].detach().cpu().numpy(), 
                                    title=f"Water depth at time instant {time_instant}", 
                                    vmin=colorbar_vmin['water_depth'], 
                                    vmax=colorbar_vmax['water_depth'])
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    frames.append(imageio.v2.imread(buf))
    buf.close()
    plt.close(fig)
imageio.mimsave(os.path.join(figure_dir, f"reconstruction_water_depth_forecast_test_0.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 1],
#                                     target[0, time_index, 1], 
#                                     title=f"Discharge x at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_x'], 
#                                     vmax=colorbar_vmax['discharge_x'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_x_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 2],
#                                     target[0, time_index, 2], 
#                                     title=f"Discharge y at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_y'], 
#                                     vmax=colorbar_vmax['discharge_y'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_y_forecast.gif"), frames, duration=1.0, loop=0)

# # %%


In [ ]:
mean = mean.cpu().numpy()

In [ ]:
from train_utils import plot_prediction_truth_error
relative_l2_error = compute_relative_l2_error(dec.detach().cpu().numpy(), target.detach().cpu().numpy())

frames = []
time_index = 0
fig = plot_prediction_truth_error(dec[0, time_index, 0].detach().cpu().numpy(),
                                target[0, time_index, 0].detach().cpu().numpy(), 
                                title=f"Water depth at time instant {time_instant}", 
                                vmin=colorbar_vmin['water_depth'], 
                                vmax=colorbar_vmax['water_depth'])
plt.savefig(VAE_FIGURE_DIR / "water_depth_time0.png")
plt.close(fig)

In [ ]:
frames = []
for time_instant in range(1, num_time_instants + 1):
    time_index = time_instant - 1
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))

    im00 = ax[0, 0].imshow(mean[time_index][0], cmap='viridis')
    ax[0, 0].set_title('Channel 1')
    fig.colorbar(im00, ax=ax[0, 0], pad=0.05, orientation='horizontal')

    im01 = ax[0, 1].imshow(mean[time_index][1], cmap='viridis')
    ax[0, 1].set_title('Channel 2')
    fig.colorbar(im01, ax=ax[0, 1], pad=0.05, orientation='horizontal')

    im02 = ax[0, 2].imshow(mean[time_index][2], cmap='viridis')
    ax[0, 2].set_title('Channel 3')
    fig.colorbar(im02, ax=ax[0, 2], pad=0.05, orientation='horizontal')

    im03 = ax[0, 3].imshow(mean[time_index][3], cmap='viridis')
    ax[0, 3].set_title('Channel 4')
    fig.colorbar(im03, ax=ax[0, 3], pad=0.05, orientation='horizontal')


    im10 = ax[1, 0].imshow(mean[time_index][4], cmap='viridis')
    ax[1, 0].set_title('Channel 5')
    fig.colorbar(im10, ax=ax[1, 0], pad=0.05, orientation='horizontal')

    im11 = ax[1, 1].imshow(mean[time_index][5], cmap='viridis')
    ax[1, 1].set_title('Channel 6')
    fig.colorbar(im11, ax=ax[1, 1], pad=0.05, orientation='horizontal')

    im12 = ax[1, 2].imshow(mean[time_index][6], cmap='viridis')
    ax[1, 2].set_title('Channel 7')
    fig.colorbar(im12, ax=ax[1, 2], pad=0.05, orientation='horizontal')

    im13 = ax[1, 3].imshow(mean[time_index][7], cmap='viridis')
    ax[1, 3].set_title('Channel 8')
    fig.colorbar(im13, ax=ax[1, 3], pad=0.05, orientation='horizontal')

    fig.suptitle(f"Latent variables at time instant {time_instant}")
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    frames.append(imageio.v2.imread(buf))
    buf.close()
    plt.close(fig)
imageio.mimsave(VAE_FIGURE_DIR / f"latent_variables_{checkpoint_epoch}.gif", frames, duration=1.0, loop=0)

In [ ]:
figure_dir = VAE_FIGURE_DIR
figure_dir.mkdir(parents=True, exist_ok=True)

from train_utils import plot_prediction_truth_error_relative
relative_l2_error = compute_relative_l2_error(dec.detach().cpu().numpy(), target.detach().cpu().numpy())
print(relative_l2_error)
frames = []
for time_instant in range(1, num_time_instants + 1):
    time_index = time_instant - 1
    fig = plot_prediction_truth_error_relative(dec[0, time_index, 0].detach().cpu().numpy(),
                                    target[0, time_index, 0].detach().cpu().numpy(), 
                                    title=f"Water depth at time instant {time_instant}", 
                                    vmin=colorbar_vmin['water_depth'], 
                                    vmax=colorbar_vmax['water_depth'])
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    frames.append(imageio.v2.imread(buf))
    buf.close()
    plt.close(fig)
imageio.mimsave(os.path.join(figure_dir, f"reconstruction_water_depth_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 1],
#                                     target[0, time_index, 1], 
#                                     title=f"Discharge x at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_x'], 
#                                     vmax=colorbar_vmax['discharge_x'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_x_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 2],
#                                     target[0, time_index, 2], 
#                                     title=f"Discharge y at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_y'], 
#                                     vmax=colorbar_vmax['discharge_y'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_y_forecast.gif"), frames, duration=1.0, loop=0)

# # %%
